# NB1 - Cold, warm, decaying or ultralight dark matter?
Tonale school, *Multimessenger cosmology*: Lectures 1 and 2 of the working-group material.

1. Run `CLASS` from Python and plot the matter power spectrum $P(k,z)$ and the growth factor.
2. Watch single modes grow: the Mészáros effect.
3. Translate wavenumbers into halo masses.
4. What if there were no dark matter? The CMB and the matter power spectrum.
5. Go beyond cold dark matter: warm, ultralight and decaying dark matter (and massive neutrinos, for comparison).
6. **Open exercise: given an observed power spectrum, what could it be?**

**No internet at the school**: install the code beforehand. The notebook is written for **AxiCLASS**, which is CLASS
(v3.3) with a scalar field: `git clone https://github.com/PoulinV/AxiCLASS`, then `make` in that folder (it builds the
Python module `classy`). The standard `pip install classy` also works: only the ultralight field of section 5 then
uses precomputed arrays. Without any `classy`, every plot is drawn from the precomputed arrays of `data/nb1.npz`,
with a coarser choice of model parameters.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

D = np.load("data/nb1.npz")          # precomputed arrays and the mystery spectra
KH = D["kh"]                         # wavenumbers in h/Mpc, used everywhere below
try:                                  # CLASS is used only if it is installed and passes a quick test
    from classy import Class
    _c = Class(); _c.set({"h": 0.6736, "omega_b": 0.02237, "omega_cdm": 0.12, "YHe": 0.2454}); _c.compute()
    HAVE_CLASS = abs(_c.Neff() - 3.044) < 0.05          # some installations of classy return N_eff = 0
except Exception:
    HAVE_CLASS = False
if not HAVE_CLASS:
    print("classy is missing or fails a quick test (N_eff should be 3.044): "
          "using the precomputed arrays of data/nb1.npz")
# options of AxiCLASS for a scalar field with V = m^2 phi^2 / 2 (followed as a fluid once it oscillates)
ULA = {"scf_potential": "axionquad", "scf_tuning_index": 1, "axionquad_mass_is_log10": "no", "attractor_ic_scf": "no",
       "scf_evolve_as_fluid": "yes", "scf_evolve_like_axionCAMB": "yes", "threshold_scf_fluid_m_over_H": 3,
       "do_shooting": "yes", "do_shooting_scf": "yes", "scf_has_perturbations": "yes", "use_big_theta_scf": "no",
       "use_delta_scf_over_1plusw": "no", "include_scf_in_delta_m": "yes", "include_scf_in_delta_cb": "yes",
       "include_scf_in_growth_factor": "no", "k_per_decade_for_pk": 30}      # fine sampling in k: the cutoff is sharp
try:                                  # is this classy AxiCLASS, i.e. does it know the scalar field?
    _c = Class(); _c.set(dict(ULA, h=0.6736, omega_b=0.02237, omega_cdm=0.11, Omega_scf=0.02,
                              scf_parameters="1e-24,0.1,0")); _c.compute()
    HAVE_AXI = HAVE_CLASS
except Exception:
    HAVE_AXI = False
if HAVE_CLASS and not HAVE_AXI:
    print("this classy is the standard CLASS, not AxiCLASS: the ultralight field uses the precomputed arrays")
plt.rcParams.update({"figure.figsize": (7, 4.5), "font.size": 13})

## 0. Running CLASS
`run(extra)` computes one cosmology (the Planck 2018 best fit, modified by the dictionary `extra`) and
`pk(c, z)` returns the **linear** matter power spectrum in $(h^{-1}{\rm Mpc})^3$ on the grid `KH`.

In [ ]:
BASE = {"h": 0.6736, "omega_b": 0.02237, "omega_cdm": 0.1200, "A_s": 2.1e-9, "n_s": 0.9649,
        "tau_reio": 0.0544, "N_ur": 2.0328, "N_ncdm": 1, "m_ncdm": 0.06}

def run(extra=None, kmax=35.0, zmax=10.5):
    p = dict(BASE, output="mPk")
    p.update({"P_k_max_h/Mpc": kmax, "z_max_pk": zmax})
    p.update(extra or {})
    c = Class(); c.set(p); c.compute()
    return c

def pk(c, z=0.0):
    h = c.h()
    return np.array([c.pk_lin(k * h, z) for k in KH]) * h**3

if HAVE_CLASS:
    c0 = run()                       # the reference LCDM model, kept for the whole notebook
    print("sigma8 = %.3f, Omega_m = %.3f" % (c0.sigma8(), c0.Omega_m()))

def pk_ref(z):
    "LCDM spectrum at z (CLASS, or the precomputed table at z = 0, 0.5, 1, 3)."
    return pk(c0, z) if HAVE_CLASS else D["pk_z%g" % z]

## 1. The $\Lambda$CDM power spectrum and its growth (Lecture 1)
**Look at**
1. $P(k)$ at $z=0,1,3$. Where is the maximum? Compare with $k_{\rm eq}=0.073\,\omega_m\,{\rm Mpc}^{-1}$.
2. The logarithmic slope $d\ln P/d\ln k$: $n_s$ on large scales, tending to $n_s-4$ (up to a logarithm) on small scales.
3. The ratio $P(k,z)/P(k,0)$ does not depend on $k$: it is $D^2(z)$. Why is $(1+z)\,D(z)$ constant at high redshift,
   and why does it drop at $z\lesssim1$?

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
for z in (0.0, 1.0, 3.0):
    ax[0].loglog(KH, pk_ref(z), label="z = %g" % z)
ax[0].set_xlabel("k [h/Mpc]"); ax[0].set_ylabel(r"$P(k)$ [(Mpc/h)$^3$]"); ax[0].legend()

ax[1].semilogx(KH, np.gradient(np.log(pk_ref(0.0)), np.log(KH)))
for y in (BASE["n_s"], BASE["n_s"] - 4):
    ax[1].axhline(y, color="k", ls=":")
ax[1].set_xlabel("k [h/Mpc]"); ax[1].set_ylabel(r"$d\ln P/d\ln k$")

if HAVE_CLASS:
    zg = np.linspace(0, 10, 41)
    Dg = np.sqrt([c0.pk_lin(0.05, z) / c0.pk_lin(0.05, 0.0) for z in zg])
else:
    zg, Dg = D["growth_z"], D["growth_D"]
ax[2].plot(zg, (1 + zg) * Dg)
ax[2].set_xlabel("z"); ax[2].set_ylabel(r"$(1+z)\,D(z)/D(0)$");

## 2. One mode at a time: the Mészáros effect (Lecture 1)
$|\delta_{\rm cdm}(a)|$ for a few wavenumbers (CLASS: `k_output_values`, Newtonian gauge).
Identify (i) the super-horizon plateau, (ii) the logarithmic growth during radiation domination,
(iii) $\delta\propto a$ during matter domination. Which mode has grown the least between its horizon entry and today,
and what does that do to $P(k)$?

In [ ]:
ks = [0.002, 0.05, 0.3, 2.0]          # 1/Mpc
if HAVE_CLASS:
    c = Class()
    c.set(dict(BASE, output="mPk", k_output_values=",".join(map(str, ks)), gauge="newtonian"))
    c.compute()
    pert = c.get_perturbations()["scalar"]
    curves = [(pert[i]["a"], np.abs(pert[i]["delta_cdm"]) / np.abs(pert[i]["delta_cdm"][0])) for i in range(len(ks))]
else:
    curves = [(D["mesz_a_k%g" % k], D["mesz_d_k%g" % k]) for k in ks]
for k, (a, d) in zip(ks, curves):
    plt.loglog(a, d, label="k = %g/Mpc" % k)
plt.axvline(1 / 3400.0, color="k", ls=":"); plt.text(1 / 3000.0, 2, "equality")
plt.xlabel("a"); plt.ylabel(r"$|\delta_{\rm cdm}|$ / initial value"); plt.legend();

## 3. From wavenumbers to halo masses (Lecture 1)
The r.m.s. fluctuation $\sigma(M,z)$ for $M=10^{9}$ and $10^{12}\,M_\odot$, with the Lagrangian radius
$R=(3M/4\pi\bar\rho_m)^{1/3}$. Above which redshift is each scale linear ($\sigma<1$)? Which probes reach it?

In [ ]:
if HAVE_CLASS:
    rho_m = 2.775e11 * c0.Omega_m() * c0.h()**2          # Msun / Mpc^3
    zz = np.linspace(0, 8, 33)
    sig = {M: np.array([c0.sigma((3 * M / (4 * np.pi * rho_m))**(1 / 3), z) for z in zz]) for M in (1e9, 1e12)}
else:
    zz = D["sig_z"]; sig = {M: D["sig_M%.0e" % M] for M in (1e9, 1e12)}
for M in sig:
    plt.semilogy(zz, sig[M], label=r"$M=10^{%d}\,M_\odot$" % np.log10(M))
plt.axhline(1, color="k", lw=0.8); plt.xlabel("z"); plt.ylabel(r"$\sigma(M,z)$"); plt.legend();

## 4. What if there were no dark matter? (Lecture 1)
### The CMB (question 1.2)
A universe without dark matter, in which **all the matter is baryonic** (same $\omega_m$), next to $\Lambda$CDM and
to the Planck data.

**Explain what you see**
1. The heights of the peaks: why is the first peak so high without dark matter, and what happens to the following
   ones? In which universe is the third peak as high as the second?
2. The positions of the peaks: why do they move, although $h$ and $\omega_m$ are the same in the two universes?
3. Which universe do the data choose? Could you rescue the other one by changing $h$?

In [ ]:
NO_DM = {"no dark matter: all the matter is baryonic": {"omega_b": 0.1424, "omega_cdm": 1e-10, "YHe": 0.2454}}
ELL = np.arange(2, 2501)

def dl_tt(extra=None):
    "Lensed temperature spectrum l(l+1)C_l/2pi in muK^2 on the grid ELL."
    p = dict(BASE, output="tCl,pCl,lCl", lensing="yes", l_max_scalars=2500)
    p.update(extra or {})
    c = Class(); c.set(p); c.compute()
    return ELL * (ELL + 1) * c.lensed_cl(2500)["tt"][2:] / (2 * np.pi) * 2.7255e6**2

plt.figure(figsize=(9, 4.8))
plt.errorbar(D["planck_ell"], D["planck_dl"], D["planck_err"], fmt="k.", ms=5, label="Planck 2018")
plt.plot(ELL, dl_tt() if HAVE_CLASS else D["cl_lcdm"], "k", lw=1, label=r"$\Lambda$CDM")
for i, (name, extra) in enumerate(NO_DM.items()):
    plt.plot(ELL, dl_tt(extra) if HAVE_CLASS else D["cl_nodm_%d" % i], label=name)
plt.yscale("log"); plt.ylim(50, 4e4); plt.xlim(2, 2000)
plt.xlabel(r"multipole $\ell$"); plt.ylabel(r"$\ell(\ell+1)C_\ell^{TT}/2\pi$ [$\mu$K$^2$]"); plt.legend();

### The matter power spectrum (question 1.3)
The same two universes. Left: the matter power spectrum today. Right: the r.m.s. fluctuation $\sigma(M)$ of section 3,
today; an object of mass $M$ exists once $\sigma(M)\gtrsim1$.

**Explain what you see**
1. The oscillations of the baryonic universe. Where do they come from? Compare their spacing with $2\pi/r_s$, where
   $r_s\simeq147$ Mpc is the sound horizon ($0.063\,h$/Mpc). Why are they reduced to a few per cent in $\Lambda$CDM?
2. The loss of power at $k\gtrsim0.2\,h$/Mpc. What erases the small scales before recombination?
3. Which objects have formed in each universe: clusters ($10^{15}M_\odot$), the Milky Way ($10^{12}$), dwarfs ($10^{9}$)?

In [ ]:
MASSES = np.logspace(8, 15, 15)

def sigma_m(c):
    rho = 2.775e11 * c.Omega_m() * c.h()**2
    return np.array([c.sigma((3 * M / (4 * np.pi * rho))**(1 / 3), 0.0) for M in MASSES])

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].loglog(KH, pk_ref(0.0), "k", label=r"$\Lambda$CDM")
ax[1].loglog(MASSES, sigma_m(c0) if HAVE_CLASS else D["sigM_lcdm"], "k")
for i, (name, extra) in enumerate(NO_DM.items()):
    if HAVE_CLASS:
        c = run(extra); p_nodm, s_nodm = pk(c, 0.0), sigma_m(c)
    else:
        p_nodm, s_nodm = D["pk_nodm_%d" % i], D["sigM_nodm_%d" % i]
    ax[0].loglog(KH, p_nodm, label=name); ax[1].loglog(MASSES, s_nodm)
ax[0].set_xlim(1e-3, 3); ax[0].set_ylim(1e-3, 1e5); ax[0].set_xlabel("k [h/Mpc]")
ax[0].set_ylabel(r"$P(k)$ [(Mpc/h)$^3$], z = 0"); ax[0].legend()
ax[1].axhline(1, color="grey", lw=0.8); ax[1].set_xlabel(r"$M$ [$M_\odot$]"); ax[1].set_ylabel(r"$\sigma(M)$ today");

## 5. Beyond cold dark matter (Lecture 2)
Growth needs gravity to win. Four ways to weaken it, each with its own signature in $P/P_{\Lambda{\rm CDM}}$:

| mechanism | example | function | parameter |
|---|---|---|---|
| free streaming | thermal warm dark matter | `ratio("wdm", m, z)` | mass in keV |
| quantum pressure | ultralight (fuzzy) dark matter | `ratio_fdm(m22, z)` | mass in $10^{-22}$ eV |
| less matter at late times | dark matter decaying into dark radiation | `ratio_ddm(tau, z)` | lifetime in Gyr |
| a smooth component | massive neutrinos (Lecture 3) | `ratio("nu", m, z)` | $\sum m_\nu$ in eV (at fixed $\omega_m$) |

Warm dark matter and neutrinos run CLASS. The ultralight field is a scalar field with $V=\frac12m^2\phi^2$ that makes
up all the dark matter, computed with AxiCLASS (`ratio_fdm_fit` is the fitting formula of Hu, Barkana & Gruzinov 2000,
for comparison). The decay
is a 15-line growth solver: read it, it is the growth equation with a matter density that decays (CLASS can also do
it, with `omega_ini_dcdm` and `Gamma_dcdm`).

In [ ]:
T_NU = 0.71611                       # T_ncdm / T_gamma of a standard neutrino
OMEGA_R = 4.18e-5                    # photons + massless neutrinos

def p_nu(sum_mnu):                   # one massive neutrino, at fixed omega_m (omega_cdm lowered)
    return {"m_ncdm": sum_mnu, "omega_cdm": 0.12 - (sum_mnu - 0.06) / 93.14}

def p_wdm(m_kev):                    # all the dark matter is a thermal relic: a second, colder "ncdm" species
    t_w = T_NU * (93.14 * 0.12 / (1e3 * m_kev))**(1 / 3)
    return {"N_ncdm": 2, "m_ncdm": "0.06,%g" % (1e3 * m_kev), "T_ncdm": "%.5f,%.5f" % (T_NU, t_w),
            "omega_cdm": 1e-10, "omega_ncdm": "%.6f,0.12" % (0.06 / 93.14)}

def p_ula(m22):                      # all the dark matter is a scalar field with V = m^2 phi^2 / 2 (AxiCLASS)
    return dict(ULA, omega_cdm=1e-5, Omega_scf=0.12 / BASE["h"]**2, scf_parameters="%g,0.1,0" % (1e-22 * m22))

MODELS = {"nu": p_nu, "wdm": p_wdm, "ula": p_ula}
_cache = {}

def ratio(model, value, z=0.0):
    "P / P_LCDM on the grid KH. Without classy (or AxiCLASS for 'ula'): nearest precomputed value and redshift."
    if HAVE_CLASS and (model != "ula" or HAVE_AXI):
        if (model, value) not in _cache:
            _cache[(model, value)] = run(MODELS[model](value))
        return pk(_cache[(model, value)], z) / pk_ref(z)
    vals, zs = D["grid_%s_values" % model], np.array([0.0, 0.5, 1.0, 3.0])
    v, zn = vals[np.argmin(np.abs(np.log(vals / value)))], zs[np.argmin(np.abs(zs - z))]
    if v != value or zn != z:
        print("(precomputed %s = %g at z = %g)" % (model, v, zn))
    return D["grid_%s_%g_z%g" % (model, v, zn)]

def ratio_fdm(m22, z=0.0):
    "Ultralight (fuzzy) dark matter with AxiCLASS, m22 = m / 1e-22 eV."
    return ratio("ula", m22, z)

def ratio_fdm_fit(m22):
    "The fitting formula of Hu, Barkana & Gruzinov (2000): independent of redshift."
    x = 1.61 * m22**(1 / 18) * KH * BASE["h"] / (9.0 * np.sqrt(m22))
    return (np.cos(x**3) / (1 + x**8))**2

def solve_growth(tau_gyr=None, z=0.0, a_i=1e-3):
    "(delta, H, matter density) at redshift z; tau_gyr = lifetime of dark matter decaying into dark radiation."
    ob, oc = BASE["omega_b"], BASE["omega_cdm"] + 0.06 / 93.14
    ol = BASE["h"]**2 - ob - oc - OMEGA_R
    gamma = 0.0 if tau_gyr is None else 9.7779 / tau_gyr          # decay rate in units of 100 km/s/Mpc

    def rhs(lna, y):                                              # y = (rho_dm, rho_dr, delta, d delta / d ln a)
        a = np.exp(lna); rdm, rdr, d, dp = y
        hub = np.sqrt(ob / a**3 + rdm + rdr + OMEGA_R / a**4 + ol)
        dlnh = (-3 * ob / a**3 - 3 * rdm - 4 * rdr - 4 * OMEGA_R / a**4) / (2 * hub**2)
        return [-3 * rdm - gamma / hub * rdm, -4 * rdr + gamma / hub * rdm, dp,
                -(2 + dlnh) * dp + 1.5 * (ob / a**3 + rdm) / hub**2 * d]

    a_eq = OMEGA_R / (ob + oc)                                    # growing mode: delta ~ a + 2 a_eq / 3
    sol = solve_ivp(rhs, [np.log(a_i), -np.log(1 + z)], [oc / a_i**3, 0.0, a_i + 2 * a_eq / 3, a_i],
                    rtol=1e-8, atol=1e-12)
    rdm, rdr, d, dp = sol.y[:, -1]
    rho_m = ob * (1 + z)**3 + rdm                                 # baryons + surviving dark matter
    return d, np.sqrt(rho_m + rdr + OMEGA_R * (1 + z)**4 + ol), rho_m

def growth(tau_gyr=None, z=0.0):
    return solve_growth(tau_gyr, z)[0]

def hubble(tau_gyr=None, z=0.0):
    return solve_growth(tau_gyr, z)[1]

def ratio_ddm(tau_gyr, z=0.0):
    "P/P_LCDM of the density contrast: what galaxy clustering and the Lyman-alpha forest see."
    return (growth(tau_gyr, z) / growth(None, z))**2 * np.ones_like(KH)

def lensing_ddm(tau_gyr, z=0.0):
    "The same for the gravitational potential, (rho_m delta)^2: what lensing sees."
    return ratio_ddm(tau_gyr, z)[0] * (solve_growth(tau_gyr, z)[2] / solve_growth(None, z)[2])**2

print("check of the growth solver: D(z=0)/a = %.3f (LCDM: about 0.79)" % growth())

**Look at**
1. For each family, describe the signature: which scales, how sharp, any dependence on redshift?
2. Warm and ultralight dark matter both cut the spectrum. Compare the scale of the cut with your estimates of the
   free-streaming horizon and of the Jeans scale (Lecture 2). What distinguishes the two?
3. Neutrinos: read the plateau and compare with $-8f_\nu$, $f_\nu=\sum m_\nu/(93.14\,{\rm eV}\,\omega_m)$ (Lecture 3).
4. Change `Z` to 3. Which signatures move, which do not, and why?

In [ ]:
Z = 0.0
fig, ax = plt.subplots(1, 4, figsize=(20, 4.3), sharey=True)
for m in (1.0, 2.0, 5.0):
    ax[0].semilogx(KH, ratio("wdm", m, Z), label="WDM %g keV" % m)
for m22 in (1.0, 10.0, 100.0):
    ax[1].semilogx(KH, ratio_fdm(m22), label=r"$m=%g\times10^{-22}$ eV" % m22)
for tau in (50.0, 100.0, 300.0):
    ax[2].semilogx(KH, ratio_ddm(tau, Z), label=r"$\tau=%g$ Gyr" % tau)
for m in (0.1, 0.2, 0.4):
    ax[3].semilogx(KH, ratio("nu", m, Z), label=r"$\sum m_\nu=%g$ eV" % m)
for a, t in zip(ax, ["warm (free streaming)", "ultralight (quantum pressure)", "decaying (less matter)",
                     "neutrinos (smooth component)"]):
    a.set_title(t); a.set_xlabel("k [h/Mpc]"); a.set_ylim(0, 1.1); a.legend(fontsize=10)
ax[0].set_ylabel(r"$P/P_{\Lambda{\rm CDM}}$ at z = %g" % Z);

### Check your estimates: where is the cut? (Lecture 2, question 2.2)
The wavenumber at which the power is halved, for warm and for ultralight dark matter. Compare with your estimate
of the free-streaming horizon, $k_{\rm fsh}=\pi/\lambda_{\rm fsh}$, and of the Jeans scale. How does the
cut move with the mass? Change the masses and plot `ratio("wdm", m)` yourself.

In [ ]:
def k_half(r):
    "Wavenumber [h/Mpc] at which P/P_LCDM drops below 1/2 (nan if it does not on the grid KH)."
    return KH[np.argmax(r < 0.5)] if r.min() < 0.5 else np.nan

for m in (1.0, 2.0, 3.0):
    print("warm, %g keV          : power halved at k = %4.1f h/Mpc" % (m, k_half(ratio("wdm", m))))
for m22 in (1.0, 10.0, 100.0):
    print("ultralight, %5.0e eV : power halved at k = %4.1f h/Mpc" % (m22 * 1e-22, k_half(ratio_fdm(m22))))

### One mode at a time: the density contrast of each kind of dark matter
The spectra above are the end of a story. Here is the story: $|\delta(a)|$ of the dark component itself, for a
**large-scale** mode ($k=0.05$/Mpc, above every cut) and a **small-scale** one ($k=10$/Mpc, below the cuts of warm and
ultralight dark matter), compared with cold dark matter. Bottom row: the ratio to $\delta_{\rm cdm}$ of $\Lambda$CDM.
All the runs start from the same primordial perturbation (CLASS: `k_output_values`, synchronous gauge;
the decay into dark radiation is CLASS's own `dcdm` species).

1. Large scale: which species end up following cold dark matter, and from when? What was each doing before?
2. Small scale: warm and ultralight dark matter both lose a factor of 10 to 100. When is it lost, and what happens
   afterwards? Why do neutrinos never catch up?
3. Decaying dark matter: its density drops, but what about its density *contrast*?
4. Square the ratios today and compare with $P/P_{\Lambda{\rm CDM}}$ in the four panels above.

In [ ]:
K_MODES = [0.05, 10.0]               # 1/Mpc: a large and a small scale
A_GRID = np.logspace(-6, 0, 1200)

def p_ddm(tau_gyr):                  # all the dark matter decays into dark radiation (CLASS: "dcdm")
    return {"omega_cdm": 1e-10, "omega_ini_dcdm": 0.12, "Gamma_dcdm": 977.79 / tau_gyr}      # Gamma in km/s/Mpc

SPECIES = [("cdm", "cold", {}, "delta_cdm", "k"),
           ("wdm", "warm, 1 keV", p_wdm(1.0), "delta_ncdm[1]", "C0"),
           ("ula", r"ultralight, $10^{-22}$ eV", p_ula(1.0), "delta_scf", "C1"),
           ("ddm", r"decaying, $\tau=50$ Gyr", p_ddm(50.0), "delta_dcdm", "C2"),
           ("nu", "neutrinos, 0.4 eV", p_nu(0.4), "delta_ncdm[0]", "C3")]

def delta_of_a(name, extra, column):
    "|delta(a)| of one species on A_GRID, for each mode of K_MODES."
    if HAVE_CLASS and (name != "ula" or HAVE_AXI):
        try:
            pert = run(dict(extra, k_output_values=",".join(map(str, K_MODES)))).get_perturbations()["scalar"]
            return [np.abs(np.interp(A_GRID, p["a"], p[column], left=np.nan)) for p in pert]
        except Exception:                                  # some versions of CLASS fail on one of the models
            print("(%s: this CLASS fails, using the precomputed table)" % name)
    return [D["dlt_%s_k%g" % (name, k)] for k in K_MODES]

deltas = {name: delta_of_a(name, extra, column) for name, label, extra, column, color in SPECIES}
fig, ax = plt.subplots(2, 2, figsize=(15, 8.5), sharex=True, gridspec_kw={"height_ratios": [3, 2]})
for i, k in enumerate(K_MODES):
    for name, label, extra, column, color in SPECIES:
        ax[0, i].loglog(A_GRID, deltas[name][i], color=color, lw=2.5 if name == "cdm" else 1.5, label=label)
        if name != "cdm":
            ax[1, i].loglog(A_GRID, deltas[name][i] / deltas["cdm"][i], color=color)
        print("k = %5g/Mpc  %-28s delta/delta_cdm today = %.3g" % (k, label, deltas[name][i][-1] / deltas["cdm"][i][-1]))
    ax[0, i].set_title("k = %g/Mpc (%s scale)" % (k, "large" if i == 0 else "small"))
    ax[0, i].set_ylim(1e-3, 3e5); ax[1, i].set_ylim(1e-4, 3); ax[1, i].axhline(1, color="k", lw=0.8)
    ax[1, i].set_xlabel("a"); ax[1, i].set_xlim(1e-6, 1)
    for b in ax[:, i]:
        b.axvline(1 / 3400.0, color="k", ls=":")
ax[0, 0].text(1 / 3000.0, 3e-3, "equality"); ax[0, 0].legend(fontsize=11)
ax[0, 0].set_ylabel(r"$|\delta|$ (per unit primordial curvature)"); ax[1, 0].set_ylabel(r"$|\delta|/|\delta_{\rm cdm}|$");

### A scalar field: dark energy, then dark matter (Lecture 2, question 2.3)
A field with $V=\frac12m^2\phi^2$ in an expanding Universe: $\ddot\phi+3H\dot\phi+m^2\phi=0$, solved here as a spectator in
the $\Lambda$CDM background, in a few lines (left and middle): the integration stops a few oscillations after $m=3H$,
and the density is then continued as $a^{-3}$ (dashed). AxiCLASS solves the same equation, with the field as the dark
matter and with its perturbations: this gives the power spectrum on the right.

**Look at**
1. Left: the field is frozen while $m\ll H$, then oscillates. Read the onset for each mass and compare with $m=3H$.
2. Middle: the density is constant (like a cosmological constant), then dilutes like matter. Which of these fields
   can be the dark matter at equality? Which one is still dark energy today?
3. Right: the matter power spectrum for three masses. Compare the cut with your de Broglie estimate of the Jeans scale.
4. Change the masses. Which is the lightest field that behaves as matter at recombination?

In [ ]:
H0_EV = 2.133e-33 * BASE["h"]                                     # H0 in eV
OM_R = OMEGA_R / BASE["h"]**2
OM_M = (BASE["omega_b"] + BASE["omega_cdm"] + 0.06 / 93.14) / BASE["h"]**2

def hubble_lcdm(a):
    "H / H0"
    return np.sqrt(OM_R / a**4 + OM_M / a**3 + 1 - OM_M - OM_R)

def scalar_field(m_ev, stop=100.0):
    "a, phi/phi_i, rho/rho_i and w = p/rho for V = m^2 phi^2 / 2; stops when m/H = stop (or today)."
    mu = m_ev / H0_EV
    grid = np.logspace(-9, 0, 4000)
    r = mu / hubble_lcdm(grid)                                    # m/H, grows with time
    a0 = grid[np.argmax(r > 1e-3)]
    a1 = grid[np.argmax(r > stop)] if r[-1] > stop else 1.0

    def rhs(N, y):                                                # y = (phi, d phi / d ln a)
        a = np.exp(N); E = hubble_lcdm(a)
        dlnE = (-4 * OM_R / a**4 - 3 * OM_M / a**3) / (2 * E**2)
        return [y[1], -(3 + dlnE) * y[1] - (mu / E)**2 * y[0]]

    N = np.linspace(np.log(a0), np.log(a1), 4000)
    phi, dphi = solve_ivp(rhs, [N[0], N[-1]], [1.0, 0.0], t_eval=N, rtol=1e-8, atol=1e-10).y
    a = np.exp(N); kin, pot = (hubble_lcdm(a) * dphi)**2, (mu * phi)**2
    return a, phi, (kin + pot) / mu**2, (kin - pot) / (kin + pot)

fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
for m in (1e-33, 1e-30, 1e-27, 1e-22):
    a, phi, rho, w = scalar_field(m)
    lab = r"$m=10^{%d}$ eV" % np.log10(m)
    ax[0].semilogx(a, phi, label=lab)
    line, = ax[1].loglog(a, rho, label=lab)
    if a[-1] < 1:
        ax[1].loglog([a[-1], 1], [rho[-1], rho[-1] * a[-1]**3], "--", color=line.get_color())
    i3 = np.argmax(m / H0_EV / hubble_lcdm(a) > 3)
    kin, pot = rho * (1 + w) / 2, rho * (1 - w) / 2                 # averaged over the last quarter of the run
    w_end = (kin[-1000:].mean() - pot[-1000:].mean()) / rho[-1000:].mean()
    print("m = %.0e eV: m = 3H at 1+z = %8.3g; <w> at the end of the run = %+.2f" % (m, 1 / a[i3] if i3 else np.nan, w_end))
for x in ax[:2]:
    x.axvline(1 / 3400.0, color="grey", ls=":"); x.set_xlabel("a")
ax[0].set_ylabel(r"$\phi/\phi_{\rm ini}$"); ax[0].legend(fontsize=10)
ax[1].set_ylabel(r"$\rho_\phi/\rho_{\phi,\rm ini}$"); ax[1].set_ylim(1e-20, 10)
for m22 in (1.0, 10.0, 100.0):
    line, = ax[2].semilogx(KH, ratio_fdm(m22), label=r"$m=%g\times10^{-22}$ eV" % m22)
    ax[2].semilogx(KH, ratio_fdm_fit(m22), "--", lw=1, color=line.get_color())
ax[2].set_xlabel("k [h/Mpc]"); ax[2].set_ylabel(r"$P/P_{\Lambda{\rm CDM}}$"); ax[2].legend(fontsize=10)
ax[2].set_title("AxiCLASS (dashed: fitting formula)", fontsize=11);

### Scale and time: the two handles (Lecture 2, question 2.4)
The same mechanisms, now as a function of **redshift** at a large and at a small scale. The decay into radiation
also changes the expansion rate, which distances (BAO, supernovae) measure without any reference to $k$.

For the decay into radiation, the dashed line is the power spectrum of the gravitational potential,
$(\bar\rho_m\delta)^2$, which is what lensing measures: the matter that has decayed no longer lenses.

**Look at**: which signatures are frozen in time, which build up? For each mechanism, which probe of the lecture's
probe map (a window in $k$ and $z$, or $H(z)$ only) is the best one, and which sees nothing?

In [ ]:
zs = [3.0, 1.0, 0.5, 0.0]
KS = (0.2, 15.0)
cases = [("WDM 2 keV", lambda z: ratio("wdm", 2.0, z)), (r"ultralight, $10^{-21}$ eV", lambda z: ratio_fdm(10.0)),
         (r"decay into radiation, $\tau=50$ Gyr", lambda z: ratio_ddm(50.0, z)),
         (r"$\sum m_\nu=0.4$ eV", lambda z: ratio("nu", 0.4, z))]
fig, ax = plt.subplots(1, 3, figsize=(17, 4.3))
for name, model in cases:
    r = np.array([model(z) for z in zs])
    for a, k in zip(ax, KS):
        a.plot(zs, [np.interp(k, KH, x) for x in r], "o-", label=name)
for a, k in zip(ax, KS):
    a.plot(zs, [lensing_ddm(50.0, z) for z in zs], "s--", color="C2",
           label=r"decay into radiation, $\tau=50$ Gyr, seen by lensing")
    a.set_title("k = %g h/Mpc" % k); a.set_xlabel("z"); a.set_ylabel(r"$P/P_{\Lambda{\rm CDM}}$"); a.set_ylim(0.3, 1.05)
ax[0].legend(fontsize=8, loc="lower right")
zz = np.linspace(0, 3, 13)
for tau in (30.0, 50.0, 100.0):
    ax[2].plot(zz, [hubble(tau, z) / hubble(None, z) for z in zz], label=r"decay into radiation, $\tau=%g$ Gyr" % tau)
ax[2].set_title(r"expansion rate (same early Universe, same $\Lambda$)"); ax[2].set_xlabel("z")
ax[2].set_ylabel(r"$H/H_{\Lambda{\rm CDM}}$"); ax[2].legend(fontsize=10);

## 6. Open exercise: given an observed power spectrum, what could it be?
Four universes, **A, B, C, D**. In each, two surveys measured the matter power spectrum and published its ratio
to the $\Lambda$CDM prediction (the one fitted to the CMB):

- a galaxy survey at $z=0.5$, on $0.02<k<0.6\,h$/Mpc, with 2% errors;
- a small-scale probe (think Lyman-$\alpha$ forest) at $z=3$, on $0.5<k<25\,h$/Mpc, with 6% errors.

In each universe the dark sector differs from $\Lambda$CDM in **one** way, taken from the table of section 5
(warm, ultralight, decaying into radiation, or a massive neutrino).

**For each universe**
1. Look before you fit: is the deviation scale dependent? Redshift dependent? Sharp or smooth?
2. Which mechanism do you suspect? Estimate its parameter by hand from the scale or the amplitude of the effect,
   **then** check with `show(...)` and `chi2(...)`: a good fit has $\chi^2\approx30$ for 30 points.
3. Play the devil's advocate: find a **second** explanation that also fits (another family, or plain $\Lambda$CDM with
   a different amplitude: write the ratio yourself). How bad is its $\chi^2$?
4. Which additional messenger would settle it (CMB lensing, a third redshift, smaller scales, distances, counts of
   small halos...)?

In [ ]:
def mystery(label):
    "The two data sets of universe 'A', 'B', 'C' or 'D': a list of (z, k, ratio, error)."
    return [(float(D["myst_z%s" % t]), D["myst_%s_k%s" % (label, t)], D["myst_%s_r%s" % (label, t)],
             D["myst_%s_e%s" % (label, t)]) for t in "12"]

def chi2(label, model):
    "model: a function of z returning P/P_LCDM on the grid KH."
    return sum(np.sum(((np.interp(k, KH, model(z)) - r) / e)**2) for z, k, r, e in mystery(label))

def show(label, models=()):
    "Plot universe `label` with a list of candidate models: (name, function of z)."
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
    for a, (z, k, r, e) in zip(ax, mystery(label)):
        a.errorbar(k, r, e, fmt="ko", ms=4)
        for name, model in models:
            a.semilogx(KH, model(z), label="%s ($\\chi^2$ = %.0f)" % (name, chi2(label, model)))
        a.axhline(1, color="grey", lw=0.8); a.set_xscale("log")
        a.set_xlim(0.7 * k.min(), 1.4 * k.max()); a.set_ylim(max(-0.08, r.min() - 0.15), 1.15)
        a.set_xlabel("k [h/Mpc]"); a.set_title("universe %s, z = %g" % (label, z))
        if models: a.legend(fontsize=10)
    ax[0].set_ylabel(r"$P_{\rm obs}/P_{\Lambda{\rm CDM}}$")

for label in "ABCD":
    show(label)

An example of a test (it is **not** the answer): is universe A just $\Lambda$CDM with a lower amplitude,
or does it contain a 0.1 eV neutrino?

In [ ]:
show("A", [("LCDM, amplitude lower by 10%", lambda z: 0.9 * np.ones_like(KH)),
           ("neutrino 0.1 eV", lambda z: ratio("nu", 0.1, z))])

### To discuss together
- Two of the four universes show a cut at nearly the same wavenumber. What exactly tells them apart, and would
  real data (errors, non-linearities, the thermal state of the gas) let you see it?
- One universe looks like $\Lambda$CDM at $z=3$ and not at $z=0.5$. Which other simple change gives a
  scale-independent offset, and how do the two redshifts, or a distance measurement, separate them?
- These spectra are linear. Which of your conclusions would survive at $z=0$ on $k>1\,h$/Mpc, where structure
  formation is non-linear and baryonic feedback moves the spectrum by 10 to 30%?